# Tic Tac Toe with Competing Agents

## Scenario

This experiment is distinct from the previous 'competing drive' notebooks.

This time I wanted to attempt to get an agent to play a game of Tic Tac Toe without any explicitly encoded rules, purely through preference alignment.

Furthermore, I wanted it to play against a copy of itself.

Being a game of perfect information this seemed like a reasonable challenge, with a view towards modelling partial information games like poker in the future.

I was successful, with some caveats:
1. It works perfectly on a 3 * 3 grid. Moving to 4*4 I see some non-optimal moves but I think that parameter tuning could fix this.
2. It takes a couple of seconds to run a full 3 * 3 grid game, but a 4 * 4 takes 5 minutes **per move** due to moving from 3^9 to 3^16 possible cell state combinations.

Parameter tuning is slow because I have to use a 4 * 4 grid to evaluate changes. If I can speed up the inference, I can experiment with different settings quickly.

In the next notebook I plan to explore the JAX version of pymdp to see if it is faster.

In [13]:
%pip install inferactively-pymdp

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [14]:
import numpy as np
import matplotlib.pyplot as plt
from pymdp.agent import Agent
from pymdp import utils
from copy import deepcopy

## Hidden States

We will begin by defining a few constants to make the following code easier to read

In [15]:
grid_size = 3
win_length = 3  # e.g. 3 for classic tic tac toe
n_cells = grid_size * grid_size

# Empty Count options (0 to n_cells inclusive,so 17 vals for 16 cells)
empty_count_states = list(range(0, n_cells + 1))

CELL_EMPTY = 0
CELL_O = 1
CELL_X = 2
n_cell_states = 3  # Empty, O, X

PLAYER_O = 0
PLAYER_X = 1
n_turn_states = 2  # O's turn, X's turn

ACTION_STAY = 0
ACTION_MARK = 1
n_empty_count_states = len(empty_count_states)

num_states = [n_cell_states] * n_cells + [n_turn_states] + [n_empty_count_states]

print(f"Number of states\n{n_cell_states} cell states * {n_cells} cells, 2 turn states, {n_empty_count_states} empty states:\n{num_states}")

Number of states
3 cell states * 9 cells, 2 turn states, 10 empty states:
[3, 3, 3, 3, 3, 3, 3, 3, 3, 2, 10]
